In [ ]:
import numpy as np
from scipy.stats import norm, lognorm, multivariate_normal
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import sys
sys.path.append("..")
from src.dataset import CRKPTransmissionSimulator
from src.approx_bc import abc_rejection_sampler
from src.utils import get_results
from timeit import default_timer as timer
import matplotlib.gridspec as gs
import warnings
warnings.filterwarnings('ignore')

In [ ]:
## load data
epi_dir = '/Volumes/umms-esnitkin/Project_KPC_LTACH/Analysis/LTACH_transmission_modeling/preprocessed/resampled'

SCALE = np.array([129., 28., 38., 35., 27., 17., 2])

labels_full = ["Facility", "Floor 1", "Floor 2", "Floor 3", "Floor 4", "SCU", "Room"]

K = 100

In [ ]:
with open(f"{epi_dir}/observed_data.npy", "rb") as f:
            x_o = (np.load(f).T / SCALE).T

In [ ]:
load_abc_homog = True
# load_abc_het_b = True # going to ignore this prior
load_abc_het = True

In [ ]:
def predictive_checker(sample, model, mode="Post.", save=None, K=100, het=True, bw=False):
    predictive = np.empty((K, 7, 53))
    for i in range(K):
        x_rep = model.CRKP_simulator(sample[i], i, summarize=True, show_full=True) * np.repeat(SCALE[:, None], 53, 1)
        predictive[i] = np.array(x_rep)
    x_o_rescaled = (x_o.T * SCALE).T
    predictive_error = ((predictive - x_o_rescaled)**2).mean(0).mean(-1)
    predictive = predictive[:30]
    for j in range(7):
        p_mean = predictive.mean(0)[j]
        labels = [f"{mode} Pred. Draw"] + [None for _ in range(29)]
        I_o = np.array(x_o)[j] * SCALE[j]
        linestyle = "-." if bw else "-"
        plt.plot(I_o, label="Observed", color="k", linestyle=linestyle)
        color = "tab:blue" if bw else "b"
        plt.plot(predictive[:, j, :].T, 
                 label=labels, color=color,
                 alpha=0.1)
        color = "tab:orange" if bw else "orange"
        # alpha = 0.7 if bw else 1
        plt.plot(p_mean, label=f"{mode} Pred. Mean", linestyle="--",
                 color=color)
        plt.legend()
        if j == 6:
            ylab = "Infected Roommates"
        else:
            ylab = "Infected Patients"
        plt.ylabel(ylab)
        plt.xlabel("Weeks")
        if save:
            plt.savefig(f"images/crkp/crkp_{save}_pc_{j}.png")
        plt.show()
        if not het:
            break
    return predictive_error

# Data Visualization

In [ ]:
crkp_model = CRKPTransmissionSimulator(epi_dir, 0, 1, heterogeneous=False)
x_baseline = crkp_model.CRKP_simulator(-1e8, 100, summarize=True)

infections = pd.read_csv(f"{epi_dir}/infections.csv", index_col=0)
infections.columns=np.arange(53)

In [ ]:
# figure out who is initially infected
T = infections.shape[1]
initial = np.zeros(T)
starting_index = infections.index[infections[0] == 1]
discharged = np.zeros(len(starting_index)).astype(bool)
for t in range(T):
    for i, ix in enumerate(starting_index):
        if discharged[i]:
            continue
        elif np.isnan(infections.loc[ix, t]):
            # print(ix, t)
            discharged[i] = True
    initial[t] = (1 - discharged).sum()

In [ ]:
NN = infections.notna().sum(0)
II = (infections == 1).sum(0)
sns.lineplot(NN, linestyle="--", label="Total")
sns.lineplot(II, label="Infected")
sns.lineplot(x_baseline * SCALE[0] - initial, linestyle=":", label="Screened Cases")
sns.lineplot(initial, linestyle="dashdot", label="Initial Cases")
plt.ylim(bottom=0)
plt.ylabel("Patients")
plt.xlabel("Weeks")
plt.savefig("images/crkp/crkp.png")
plt.xlim(right=53)
plt.show()

In [ ]:
np.exp(-3.5)

In [ ]:
floor_pops = []

for i in [1,2,3,4,5]:
    floor_pops.append((crkp_model.F == i).sum(0))
floor_pops = np.stack(floor_pops)

floor_cases = x_o[1:6,]

floor_pops.mean(1)

In [ ]:
labels = ["Floor 1", "Floor 2", "Floor 3", "Floor 4", "SCU"]
df_floor_pops = pd.DataFrame(floor_pops.T, columns = labels)
sns.lineplot(df_floor_pops)
plt.xlabel("Weeks")
plt.ylabel("Patients")
plt.savefig("images/crkp/floor_pops.png")
plt.show()

In [ ]:
# average case burden
for i, lab in enumerate(labels):
    temp_pop = np.where(floor_pops[i] == 0, 1, floor_pops[i])
    avg_burden = np.median(floor_cases[i] * SCALE[1:6][i] / temp_pop) # median is clearer
    print(f"{lab}: {avg_burden:.2f}")

print(np.median(II / NN))

# Prior Predictive Checks

## Homogeneous

In [ ]:
prior_mu = -2
prior_sigma = 1
crkp_model = CRKPTransmissionSimulator(epi_dir, prior_mu, prior_sigma, heterogeneous=False)
prior = norm(prior_mu)
prior_sample = prior.rvs(size=K, random_state=2)
prior_errors = predictive_checker(prior_sample, crkp_model, mode="Prior", save="homog_prior", het=False, bw=True)

## Heterogeneous

In [ ]:
prior_mu = [-3, -4, -4, -4, -4, -4, -5] 
prior_sigma = 1

Relative pairwise risks of infection by location

In [ ]:
prior_rates = np.exp(np.array(prior_mu) + prior_sigma/2)

In [ ]:
prior_risks = (np.exp(prior_mu) / SCALE) / (np.exp(prior_mu) / SCALE)[0]

In [ ]:
crkp_model = CRKPTransmissionSimulator(epi_dir, prior_mu, prior_sigma, heterogeneous=True)
prior_het = multivariate_normal(
    prior_mu, 
    np.diag(np.full((7,), prior_sigma))
)
prior_sample_het = prior_het.rvs(size=K, random_state=2)
prior_het_errors = predictive_checker(prior_sample_het, crkp_model, mode="Prior", save="prior")
print(prior_het_errors)

# MCMC

## Homogeneous

In [ ]:
mcmc_homo = np.load("mcmc_homog.npy")[:, 0]
mcmc_homo.mean(), mcmc_homo.std()

In [ ]:
np.exp(mcmc_homo).mean()

In [ ]:
prior_mu = -2
prior_sigma = 1
crkp_model = CRKPTransmissionSimulator(epi_dir, prior_mu, prior_sigma, heterogeneous=False)
mcmc_sample = np.random.choice(mcmc_homo, K, replace=False)
mcmc_homog_errors = predictive_checker(mcmc_sample, crkp_model)
mcmc_homog_errors

## Heterogeneous

In [ ]:
mcmc_het = np.load("mcmc_het2.npy")
mcmc_rates = np.exp(mcmc_het).mean(0)
mcmc_rates

In [ ]:
mcmc_risks = (mcmc_rates / SCALE) / (mcmc_rates / SCALE)[0]

In [ ]:
mcmc_het.std(0)

In [ ]:
mcmc_corr = np.corrcoef(mcmc_het.T)

plt.matshow(mcmc_corr, cmap="rocket")
plt.colorbar()
plt.xlabel(r"$\beta_j$")
plt.ylabel(r"$\beta_j$")
plt.savefig("images/crkp/corr_mcmc.png")
plt.show()

In [ ]:
prior_mu = [-3, -4, -4, -4, -4, -4, -5]
prior_sigma = 1
crkp_model = CRKPTransmissionSimulator(epi_dir, prior_mu, prior_sigma, heterogeneous=True)

idx = np.random.choice(mcmc_het.shape[0], K, replace=False)

mcmc_sample_het = mcmc_het[idx]
mcmc_het_errors = predictive_checker(mcmc_sample_het, crkp_model, save="mcmc-bw", bw=True)
mcmc_het_errors

# NPE

## Homogeneous

In [ ]:
prior_mu = -2
prior_sigma = 1
crkp_model = CRKPTransmissionSimulator(epi_dir, prior_mu, prior_sigma, heterogeneous=False)

In [ ]:
df_homo = pd.read_csv("df_crkp_homo.csv").sort_values("val_loss")

In [ ]:
mu = float(df_homo["mu"].iloc[0])
sigma = float(df_homo["sigma"].iloc[0])
print(mu, sigma)

d = norm(loc = mu, scale = sigma)
print(np.exp(d.ppf(0.05)), np.exp(d.ppf(0.95)))
print(np.exp(mu + sigma**2/2))

### Posterior Predictive Checks

In [ ]:
neural_posterior = norm(mu, sigma)
npe_sample = neural_posterior.rvs(size=K, random_state=2)
npe_homog_errors = predictive_checker(npe_sample, crkp_model)
npe_homog_errors

In [ ]:
npe_homog_errors = predictive_checker(npe_sample, crkp_model, save="homog_npe", het=False, bw=True)

## Heterogeneous

In [ ]:
i = 0
df_ff = pd.read_csv("df_crkp_het_ff.csv")
df_ff["mu"] = df_ff["mu"].apply(eval)
df_ff["sigma"] = df_ff["sigma"].apply(eval)
mu = np.array(df_ff["mu"][i][0])
sigma = np.array(df_ff["sigma"][i])
npe_rates = np.exp(mu + np.diag(sigma)/2)
npe_rates

In [ ]:
npe_risks = (npe_rates / SCALE) / (npe_rates / SCALE)[0]

In [ ]:
np.sqrt(np.diag(sigma))

In [ ]:
D = np.diag(1 / np.sqrt(np.diag(sigma)))
corr = D @ sigma @ D
plt.matshow(corr, cmap="rocket")
plt.colorbar()
plt.xlabel(r"$\beta_j$")
plt.ylabel(r"$\beta_j$")
plt.savefig("images/crkp/corr.png")
plt.show()

In [ ]:
# rough R_0 estimate (numerator)
np.sum(SCALE[1:5] * npe_rates[1:5]) / np.sum(SCALE[1:5]) + npe_rates[0] + npe_rates[-1]

### Posterior Predictive Checks

In [ ]:
prior_mu = [-3, -4, -4, -4, -4, -4, -5]
prior_sigma = 1

In [ ]:
crkp_model = CRKPTransmissionSimulator(epi_dir, prior_mu, prior_sigma, heterogeneous=True)
neural_posterior_het = multivariate_normal(mu, sigma)
# neural_posterior_het = multivariate_normal(mu, sigma)
npe_sample_het = neural_posterior_het.rvs(size=K, random_state=2)

npe_het_errors = predictive_checker(npe_sample_het, crkp_model, save="npe-bw", bw=True)
npe_het_errors

# Intervention Analysis

In [ ]:
npe_rates

In [ ]:
(npe_rates / SCALE) / (npe_rates / SCALE)[0]

## NPE

In [ ]:
pis = [
    [.2, 1, 1, 1, 1, 1, 1],
    [1, 1, 1, .1, 1, 1, 1],
    [1, 1, 1, 1, 1, 1, 1e-8],
    [1e-8, 1e-8, 1e-8, 1e-8, 1e-8, 1e-8, 1e-8]
]
L = 50
intervention = np.empty((L, len(pis), T))
case_counts = np.empty((L, len(pis)))
for j, pi in enumerate(pis):
    crkp_model = CRKPTransmissionSimulator(epi_dir, prior_mu, prior_sigma, heterogeneous=True, pi=pi, return_case_count=True)
    for i in range(L):
        x_rep, cc = crkp_model.CRKP_simulator(npe_sample_het[i], i * 2, summarize=True)
        intervention[i,j] =  np.array(x_rep)[0] * SCALE[0]
        case_counts[i, j] = cc
intervention = intervention.mean(0)
case_counts = case_counts.mean(0)

In [ ]:
case_counts # no intervention: 259

In [ ]:
# plt.plot(np.array(x_o)[0] * SCALE[0], label="Observed", color="k")
plt.plot(x_o[0] * SCALE[0], label="No Intervention", linestyle="-")
plt.plot(intervention[2], label="Room Isolation", linestyle="--")
plt.plot(intervention[1], label="Target Floor 3", linestyle=":")
plt.plot(intervention[0], label="Floor Isolation", linestyle="-.")
plt.plot(intervention[3], label="Zero Transmission", linestyle="-", alpha=0.4)
# plt.plot(intervention[3], label="Roomies", linestyle="--")
plt.legend()
plt.xlabel("Weeks")
plt.ylabel("Number Infected")
plt.savefig("images/crkp/crkp_intervention.png")
plt.show()

In [ ]:
259 - 233

In [ ]:
259 - 246

In [ ]:
df_cc = pd.DataFrame(
    data = {"Scenario": ["No Intervention", "Room Isolation", "Target Floor 3", "Floor Isolation"],
           "Total Secondary Infections": np.array([259, 245.96, 233.02, 210.68]) - 168}
)
sns.barplot(df_cc, x = "Total Secondary Infections", y = "Scenario", hue="Scenario", legend=False, alpha=0.8,
           width=0.62).set(ylabel="")
plt.savefig("images/crkp/crkp_case_count.png", bbox_inches='tight')
plt.plot()

## MCMC

In [ ]:
intervention_mcmc = np.empty((L, len(pis), T))
case_counts_mcmc = np.empty((L, len(pis)))
for j, pi in enumerate(pis):
    crkp_model = CRKPTransmissionSimulator(epi_dir, prior_mu, prior_sigma, heterogeneous=True, pi=pi, return_case_count=True)
    for i in range(L):
        x_rep, cc = crkp_model.CRKP_simulator(mcmc_sample_het[i], i * 2, summarize=True)
        intervention_mcmc[i,j] =  np.array(x_rep)[0] * SCALE[0]
        case_counts_mcmc[i, j] = cc
intervention_mcmc = intervention_mcmc.mean(0)
case_counts_mcmc = case_counts_mcmc.mean(0)

In [ ]:
case_counts_mcmc

In [ ]:
df_cc2 = pd.DataFrame(
    data = {"Scenario": ["No Intervention", "Room Isolation", "Target Floor 3", "Floor Isolation"],
           "Total Secondary Infections": np.array([259, 263.72, 266.82, 193.18]) - 168}
)
sns.barplot(df_cc2, x = "Total Secondary Infections", y = "Scenario", hue="Scenario", legend=False, alpha=0.8,
           width=0.62).set(ylabel="")
plt.savefig("images/crkp/crkp_case_count_mcmc.png", bbox_inches='tight')
plt.plot()

# ABC Comparison

## Homogeneous

In [ ]:
hetero = False

prior_mu = -2
prior_sigma = 1
crkp_model = CRKPTransmissionSimulator(epi_dir, prior_mu, prior_sigma,
                                  heterogeneous=hetero)

In [ ]:
if load_abc_homog:
    abc_sample_homog = np.load("crkp_abc_homog.npy")
else:
    prior_sampler = lambda: crkp_model.sample_logbeta(1)
    simulator = lambda theta, seed: crkp_model.CRKP_simulator(theta, seed, summarize=True)
    
    S = 100
    epsilon = .18
    abc_sample_homog, errors, raw_samples = abc_rejection_sampler(
        S, epsilon, prior_sampler, simulator, x_o[0], max_attempts=10000,
        print_every=1000
    )
    np.save("crkp_abc_homog", abc_sample_homog)

7,858 samples used.

In [ ]:
np.exp(abc_sample_homog[:, 0]).mean()

In [ ]:
abc_sample_homog[:, 0].std()

In [ ]:
np.exp(abc_sample_homog[:, 0]).mean()

In [ ]:
abc_homog_errors = predictive_checker(abc_sample_homog, crkp_model)
abc_homog_errors

## Heterogeneous

In [ ]:
hetero = True

prior_mu = [-3, -4, -4, -4, -4, -4, -5]
prior_sigma = 1


crkp_model = CRKPTransmissionSimulator(epi_dir, prior_mu, prior_sigma,
                                      heterogeneous=hetero)

if load_abc_het:
    abc_sample_het = np.load("crkp_abc_het.npy")
else:
    model = CRKPTransmissionSimulator(epi_dir, prior_mu, prior_sigma,
                                      heterogeneous=hetero)
    prior_sampler = lambda: crkp_model.sample_logbeta(1)
    simulator = lambda theta, seed: model.CRKP_simulator(theta, seed, summarize=True)
    
    S = 100
    epsilon = 9.5
    abc_sample_het, errors, raw_sample = abc_rejection_sampler(
        S, epsilon, prior_sampler, simulator, x_o, max_attempts=30000,
        print_every=1000
    )
    np.save("crkp_abc_het", abc_sample_het)

12503 samples used.

In [ ]:
abc_sample_het.mean(0)

In [ ]:
abc_sample_het.std(0)

In [ ]:
np.diag(sigma)**0.5

In [ ]:
mcmc_sample_het.std(0)

In [ ]:
abc_rates = np.exp(abc_sample_het).mean(0)
abc_risks = (abc_rates / SCALE) / (abc_rates / SCALE)[0]

In [ ]:
abc_corr = np.corrcoef(abc_sample_het.T)
plt.matshow(abc_corr, cmap="rocket")
plt.colorbar()
plt.xlabel(r"$\beta_j$")
plt.ylabel(r"$\beta_j$")
plt.savefig("images/crkp/abc_corr.png")
plt.show()

In [ ]:
abc_het_errors = predictive_checker(abc_sample_het, crkp_model, save="abc")
abc_het_errors

In [ ]:
het_table = {"Rate": labels_full, 
             "NPE Mean": np.array(rates), 
             "NPE CI": [],
            "ABC Mean": np.exp(posterior_sample).mean(0),
            "ABC CI": []}

for i in range(7):
    dist = norm(mu[i], np.sqrt(sigma[i][i]))
    het_table["NPE CI"].append(
        (float(np.exp(dist.ppf(0.05))), 
         float(np.exp(dist.ppf(0.95))))
    )
    het_table["ABC CI"].append(
        (float(np.quantile(np.exp(posterior_sample), 0.05, axis=0)[i]),
        float(np.quantile(np.exp(posterior_sample), 0.95, axis=0)[i]))
    )
het_table = pd.DataFrame(het_table)
print(het_table.to_latex(index=False))

In [ ]:
np.exp(posterior_sample).mean(0)

In [ ]:
crkp_model = CRKPTransmissionSimulator(epi_dir, prior_mu, prior_sigma, heterogeneous=True)
# neural_posterior_het = multivariate_normal(mu, sigma)
# npe_sample_het = neural_posterior_het.rvs(size=K, random_state=2)
posterior_predictive_abc = np.empty((K, 7, T))
for i in range(K):
    x_rep = crkp_model.CRKP_simulator(posterior_sample[i], i) * np.repeat(SCALE[:, None], T, 1) 
    posterior_predictive_abc[i] =  np.array(x_rep)

In [ ]:
for j in range(7):
    preds = posterior_predictive_abc[:, j, :]
    target = (x_o[j] * cap[j])
    mse = ((preds - target)**2).mean()
    print(round(mse, 3))

In [ ]:
for j in range(7):
    pp_mean = posterior_predictive_abc.mean(0)[j]
    labels = ["Post. Pred. Draw"] + [None for _ in range(K-1)]
    I_o = np.array(x_o)[j] * SCALE[j]
    plt.plot(I_o, label="Observed", color="k")
    plt.plot(posterior_predictive_abc[:, j, :].T, 
             label=labels, color="b",
             alpha=0.1)
    plt.plot(pp_mean, label="Post. Pred. Mean", linestyle="--",
             color="orange")
    # sns.lineplot(NN, color="green", linestyle="--", label="Total")
    plt.legend()
    if j == 6:
        ylab = "Infected Rooms"
    else:
        ylab = "Infected Patients"
    plt.ylabel(ylab)
    plt.xlabel("Weeks")
    # plt.savefig(f"images/crkp/crkp_ppc_abc{j}.png")
    plt.show()

# Compiling Results

## Main Descriptives

### Rates

In [ ]:
np.exp(abc_sample_het).mean(0)

In [ ]:
# MLE, 
het_rates = {"Rate": labels_full, 
             "MLE": np.array([0.112, 0.   , 0.066, 0.   , 0.   , 0.057, 0.   ]),
             "Prior": np.exp([-2.5, -3.5, -3.5, -3.5, -3.5, -3.5, -4.5]),
             "MCMC": np.exp(mcmc_het).mean(0),
             "NPE": np.array(npe_rates), 
            "ABC": np.exp(abc_sample_het).mean(0)
}
    #          for i in range(7):
    # dist = norm(mu[i], np.sqrt(sigma[i][i]))
    # het_table["NPE CI"].append(
    #     (float(np.exp(dist.ppf(0.05))), 
    #      float(np.exp(dist.ppf(0.95))))
    # )
    # het_table["ABC CI"].append(
    #     (float(np.quantile(np.exp(posterior_sample), 0.05, axis=0)[i]),
    #     float(np.quantile(np.exp(posterior_sample), 0.95, axis=0)[i]))
    # )
het_rates = pd.DataFrame(het_rates)
print(het_rates.to_latex(index=False))

### Means (log-scale)

In [ ]:
het_means = {"Rate": labels_full, 
             "Prior": [-3, -4, -4, -4, -4, -4, -5],
             "MCMC": mcmc_het.mean(0),
             "NPE": mu, 
            "ABC": abc_sample_het.mean(0)
}
print(pd.DataFrame(het_means).to_latex(index=False))

### SDs (log-scale)

In [ ]:
het_sds = {"Rate": labels_full, 
             "Prior": np.ones(7),
             "MCMC": mcmc_het.std(0),
             "NPE": np.sqrt(np.diag(sigma)), 
            "ABC": abc_sample_het.std(0)
}
print(pd.DataFrame(het_sds).to_latex(index=False))

## Relative Risks

In [ ]:
prior_risks

In [ ]:
relative_risks = pd.DataFrame(
    {
        "Transmission Zone": labels_full,
        "Prior": prior_risks,
        "MCMC": mcmc_risks,
        "NPE": npe_risks,
        "ABC": abc_risks
    }
)
print(relative_risks.to_latex(index=False))

## Predictive Error

In [ ]:
mcmc_he

In [ ]:
descriptives = ["Facility Incidence"] + [f"Floor {i} Incidence" for i in range(1, 6)] + ["Infected Rooms"]
predictive_error = pd.DataFrame(
    {
        "Descriptive": descriptives,
        "Prior": prior_het_errors,
        "MCMC": mcmc_het_errors,
        "NPE": npe_het_errors,
        "ABC": abc_het_errors
        
    }
)
print(predictive_error.to_latex(index=False))

In [ ]:
# [19.81754717,  4.32716981,  7.41792453,  8.24528302,  4.32660377,
#         2.43415094, 11.70792453]
# the homogeneous model has basically the same predictive error

In [ ]:
prior_rates = np.exp([-3, -4, -4, -4, -4, -4, -5])

In [ ]:
(prior_rates / SCALE) / (prior_rates / SCALE)[0]

In [ ]:
rates

In [ ]:
rates / SCALE

In [ ]:
SCALE

In [ ]:
(rates / SCALE) / (rates / SCALE)[0]

In [ ]:
abc_rates = np.exp(posterior_sample).mean(0)

In [ ]:
(abc_rates / SCALE) / (abc_rates / SCALE)[0]

In [ ]:
(np.exp(-2.5) / SCALE) / (np.exp(-2.5) / SCALE[0])

In [ ]:
np.log([.0352, .322, .0153, .191])

In [ ]:
np.log([.0094, .251, .0109, .154])

In [ ]:
np.log([.00820, .302, .0128, .242])

In [ ]:
-1.19 + 4.8, -1.4 + 3.4

In [ ]:
- 1.1 + 3.3

In [ ]:
-1.65 + 4.2